# EDA Block B: Feature Behavior And Signal Discovery

This notebook implements point 7 from `docs/plan.md`.

Scope:
- distribution analysis for all `last15_*` and `cumul_*` features
- correlation and redundancy mapping for numeric features
- missingness heatmaps and missingness-target association checks
- univariate signal scans with effect sizes and monotonic trends
- segment analysis by position and game context
- candidate feature shortlist generation

Protocol note:
- if `data/splits/modeling_row_folds.csv` exists, signal discovery is run on the development partition only
- holdout rows are excluded from detailed feature ranking and shortlist generation
- this notebook focuses on the main-table `last15_*` and `cumul_*` feature families because point 7 is the pre-extension feature discovery block


In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from scipy.stats import pointbiserialr, spearmanr

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
plt.style.use("ggplot")


def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").exists() and (candidate / "docs").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing `data/` and `docs/`.")


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_DIR = PROJECT_ROOT / "data"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "eda_block_b"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Artifact directory: {ARTIFACT_DIR}")


In [ ]:
main_path = DATA_DIR / "players_quarters_final.csv"
split_path = DATA_DIR / "splits" / "modeling_row_folds.csv"

main_full = pd.read_csv(main_path)
if split_path.exists():
    main_with_splits = pd.read_csv(split_path)
    analysis_main = main_with_splits[main_with_splits["partition_role"] == "development"].copy()
    print(f"Using development partition for Block B: {len(analysis_main):,} rows")
else:
    main_with_splits = main_full.copy()
    analysis_main = main_full.copy()
    print("No split artifact found. Using full main table.")

feature_cols = [
    col for col in analysis_main.columns
    if col.startswith("last15_") or col.startswith("cumul_")
]
last15_features = [col for col in feature_cols if col.startswith("last15_")]
cumul_features = [col for col in feature_cols if col.startswith("cumul_")]
context_cols = ["checkpoint", "position", "is_home", "formation", "scored_after"]

print(f"Feature count: {len(feature_cols)}")
print(feature_cols)


In [ ]:
def cohen_d_from_binary_target(series: pd.Series, target: pd.Series) -> float:
    clean = pd.DataFrame({"x": pd.to_numeric(series, errors="coerce"), "y": target}).dropna()
    pos = clean.loc[clean["y"] == 1, "x"]
    neg = clean.loc[clean["y"] == 0, "x"]
    if len(pos) < 2 or len(neg) < 2:
        return np.nan
    pooled = np.sqrt(((len(pos) - 1) * pos.var(ddof=1) + (len(neg) - 1) * neg.var(ddof=1)) / (len(pos) + len(neg) - 2))
    if pooled == 0 or np.isnan(pooled):
        return 0.0
    return float((pos.mean() - neg.mean()) / pooled)


def monotonic_trend_summary(series: pd.Series, target: pd.Series, n_bins: int = 5) -> dict:
    clean = pd.DataFrame({"x": pd.to_numeric(series, errors="coerce"), "y": target}).dropna()
    if clean["x"].nunique() < 3:
        return {
            "bin_count": 0,
            "monotonic_rho": np.nan,
            "monotonic_pvalue": np.nan,
            "bin_rate_min": np.nan,
            "bin_rate_max": np.nan,
            "monotonic_direction": "flat",
        }

    ranked = clean["x"].rank(method="average")
    bins = min(n_bins, int(clean["x"].nunique()))
    quantiles = pd.qcut(ranked, q=bins, duplicates="drop")
    grouped = clean.groupby(quantiles, observed=True)["y"].agg(["mean", "size"]).reset_index(drop=True)
    if len(grouped) < 2:
        return {
            "bin_count": 1,
            "monotonic_rho": np.nan,
            "monotonic_pvalue": np.nan,
            "bin_rate_min": float(grouped["mean"].iloc[0]),
            "bin_rate_max": float(grouped["mean"].iloc[0]),
            "monotonic_direction": "flat",
        }

    rho, pvalue = spearmanr(np.arange(len(grouped)), grouped["mean"])
    direction = "flat"
    if pd.notna(rho):
        if rho > 0.2:
            direction = "increasing"
        elif rho < -0.2:
            direction = "decreasing"

    return {
        "bin_count": int(len(grouped)),
        "monotonic_rho": float(rho) if pd.notna(rho) else np.nan,
        "monotonic_pvalue": float(pvalue) if pd.notna(pvalue) else np.nan,
        "bin_rate_min": float(grouped["mean"].min()),
        "bin_rate_max": float(grouped["mean"].max()),
        "monotonic_direction": direction,
    }


def distribution_summary(df: pd.DataFrame, features: list[str]) -> pd.DataFrame:
    rows = []
    for feature in features:
        values = pd.to_numeric(df[feature], errors="coerce")
        rows.append(
            {
                "feature": feature,
                "family": feature.split("_", 1)[0],
                "missing_pct": float(values.isna().mean() * 100),
                "zero_pct": float((values.fillna(0) == 0).mean() * 100),
                "mean": float(values.mean()),
                "std": float(values.std()),
                "median": float(values.median()),
                "p90": float(values.quantile(0.9)),
                "p99": float(values.quantile(0.99)),
                "max": float(values.max()),
            }
        )
    return pd.DataFrame(rows).sort_values(["family", "feature"]).reset_index(drop=True)


def missingness_summary(df: pd.DataFrame, features: list[str]) -> pd.DataFrame:
    rows = []
    for feature in features:
        missing_mask = df[feature].isna()
        missing_count = int(missing_mask.sum())
        non_missing_mask = ~missing_mask
        rows.append(
            {
                "feature": feature,
                "missing_count": missing_count,
                "missing_pct": float(missing_mask.mean() * 100),
                "target_rate_missing": float(df.loc[missing_mask, "scored_after"].mean()) if missing_count > 0 else np.nan,
                "target_rate_nonmissing": float(df.loc[non_missing_mask, "scored_after"].mean()) if non_missing_mask.sum() > 0 else np.nan,
            }
        )
    out = pd.DataFrame(rows)
    out["target_rate_gap"] = out["target_rate_missing"] - out["target_rate_nonmissing"]
    return out.sort_values(["missing_pct", "feature"], ascending=[False, True]).reset_index(drop=True)


def univariate_signal_summary(df: pd.DataFrame, features: list[str]) -> pd.DataFrame:
    rows = []
    target = df["scored_after"]
    for feature in features:
        values = pd.to_numeric(df[feature], errors="coerce")
        clean = pd.DataFrame({"x": values, "y": target}).dropna()
        if clean["x"].nunique() < 2:
            r_value = np.nan
            p_value = np.nan
        else:
            r_value, p_value = pointbiserialr(clean["y"], clean["x"])
        trend = monotonic_trend_summary(values, target)
        rows.append(
            {
                "feature": feature,
                "family": feature.split("_", 1)[0],
                "positive_mean": float(clean.loc[clean["y"] == 1, "x"].mean()),
                "negative_mean": float(clean.loc[clean["y"] == 0, "x"].mean()),
                "positive_median": float(clean.loc[clean["y"] == 1, "x"].median()),
                "negative_median": float(clean.loc[clean["y"] == 0, "x"].median()),
                "pointbiserial_r": float(r_value) if pd.notna(r_value) else np.nan,
                "pointbiserial_pvalue": float(p_value) if pd.notna(p_value) else np.nan,
                "cohen_d": cohen_d_from_binary_target(values, target),
                **trend,
            }
        )

    out = pd.DataFrame(rows)
    out["rate_span"] = out["bin_rate_max"] - out["bin_rate_min"]
    trend_component = np.where(
        out["monotonic_pvalue"].fillna(1.0) <= 0.10,
        0.05 * out["monotonic_rho"].abs().fillna(0),
        0.0,
    )
    out["signal_strength"] = (
        out["pointbiserial_r"].abs().fillna(0)
        + 0.5 * out["cohen_d"].abs().fillna(0)
        + trend_component
    )
    return out.sort_values("signal_strength", ascending=False).reset_index(drop=True)


def plot_heatmap(ax, matrix: pd.DataFrame, title: str, cmap: str = "coolwarm", center: float = 0.0) -> None:
    values = matrix.to_numpy(dtype=float)
    vmax = np.nanmax(np.abs(values - center)) if np.isfinite(values).any() else 1.0
    if vmax == 0:
        vmax = 1.0
    im = ax.imshow(values, aspect="auto", cmap=cmap, vmin=center - vmax, vmax=center + vmax)
    ax.set_title(title)
    ax.set_xticks(range(matrix.shape[1]))
    ax.set_xticklabels(matrix.columns, rotation=45, ha="right")
    ax.set_yticks(range(matrix.shape[0]))
    ax.set_yticklabels(matrix.index)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)


In [ ]:
dist_summary = distribution_summary(analysis_main, feature_cols)
dist_summary.to_csv(ARTIFACT_DIR / "distribution_summary.csv", index=False)

display(Markdown("## Distribution Summary For `last15_*` And `cumul_*` Features"))
display(dist_summary)

fig, axes = plt.subplots(6, 3, figsize=(16, 22))
axes = axes.flatten()
for idx, feature in enumerate(feature_cols):
    ax = axes[idx]
    values = pd.to_numeric(analysis_main[feature], errors="coerce")
    ax.hist(values.dropna(), bins=20, color="#2f6b8a")
    ax.set_title(feature)
    ax.tick_params(axis="x", labelrotation=45)
for idx in range(len(feature_cols), len(axes)):
    axes[idx].axis("off")
plt.tight_layout()
plt.show()


In [ ]:
corr_matrix = analysis_main[feature_cols].corr()
corr_pairs = (
    corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    .stack()
    .reset_index()
)
corr_pairs.columns = ["feature_a", "feature_b", "correlation"]
corr_pairs["abs_correlation"] = corr_pairs["correlation"].abs()
redundant_pairs = corr_pairs[corr_pairs["abs_correlation"] >= 0.85].sort_values("abs_correlation", ascending=False).reset_index(drop=True)
redundant_pairs.to_csv(ARTIFACT_DIR / "redundant_feature_pairs.csv", index=False)

display(Markdown("## Correlation Matrix"))
fig, ax = plt.subplots(figsize=(14, 12))
plot_heatmap(ax, corr_matrix, "Feature Correlation Heatmap")
plt.show()

display(Markdown("## Highly Correlated Feature Pairs (|r| >= 0.85)"))
display(redundant_pairs)


In [ ]:
miss_summary = missingness_summary(analysis_main, feature_cols)
miss_summary.to_csv(ARTIFACT_DIR / "missingness_summary.csv", index=False)

display(Markdown("## Missingness Summary"))
display(miss_summary)

sample_size = min(250, len(analysis_main))
missing_matrix = analysis_main[feature_cols].isna().astype(int).head(sample_size)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
plot_heatmap(axes[0], missing_matrix.T, f"Missingness Heatmap (first {sample_size} rows)", cmap="Greys", center=0.5)
axes[1].bar(miss_summary["feature"], miss_summary["missing_pct"], color="#a63d40")
axes[1].set_title("Missingness Percentage By Feature")
axes[1].set_ylabel("Missing (%)")
axes[1].tick_params(axis="x", rotation=90)
plt.tight_layout()
plt.show()


In [ ]:
signal_summary = univariate_signal_summary(analysis_main, feature_cols)
signal_summary.to_csv(ARTIFACT_DIR / "univariate_signal_summary.csv", index=False)

display(Markdown("## Univariate Signal Summary"))
display(signal_summary)

top_signal = signal_summary.head(12).iloc[::-1]
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].barh(top_signal["feature"], top_signal["pointbiserial_r"], color="#2f6b8a")
axes[0].set_title("Top Features By Point-Biserial Correlation")
axes[0].set_xlabel("point-biserial r")

axes[1].barh(top_signal["feature"], top_signal["cohen_d"], color="#6ba368")
axes[1].set_title("Top Features By Standardized Mean Difference")
axes[1].set_xlabel("Cohen's d")

plt.tight_layout()
plt.show()


In [ ]:
selected_features = []
selected_rows = []
corr_lookup = corr_matrix.abs()

for _, row in signal_summary.iterrows():
    feature = row["feature"]
    if not selected_features:
        max_abs_corr = 0.0
    else:
        max_abs_corr = float(corr_lookup.loc[feature, selected_features].max())
    if max_abs_corr >= 0.85:
        continue
    selected_features.append(feature)
    selected_rows.append(
        {
            "feature": feature,
            "family": row["family"],
            "signal_strength": float(row["signal_strength"]),
            "pointbiserial_r": float(row["pointbiserial_r"]),
            "cohen_d": float(row["cohen_d"]),
            "monotonic_rho": float(row["monotonic_rho"]),
            "rate_span": float(row["rate_span"]),
            "max_abs_corr_with_selected": max_abs_corr,
            "reason": "strong univariate signal with low redundancy at selection time",
        }
    )
    if len(selected_features) >= 8:
        break

shortlist = pd.DataFrame(selected_rows)
shortlist.to_csv(ARTIFACT_DIR / "candidate_feature_shortlist.csv", index=False)

display(Markdown("## Candidate Feature Shortlist"))
display(shortlist)


In [ ]:
top_segment_features = shortlist["feature"].head(6).tolist() if not shortlist.empty else signal_summary["feature"].head(6).tolist()


def segment_signal_matrix(df: pd.DataFrame, segment_col: str, features: list[str]) -> pd.DataFrame:
    rows = []
    for segment_value, subset in df.groupby(segment_col, dropna=False):
        result = {"segment": str(segment_value)}
        for feature in features:
            clean = pd.DataFrame(
                {
                    "target": subset["scored_after"],
                    "value": pd.to_numeric(subset[feature], errors="coerce"),
                }
            ).dropna()
            if clean["target"].nunique() < 2 or clean["value"].nunique() < 2:
                result[feature] = np.nan
            else:
                r_value, _ = pointbiserialr(clean["target"], clean["value"])
                result[feature] = float(r_value) if pd.notna(r_value) else np.nan
        rows.append(result)
    out = pd.DataFrame(rows).set_index("segment")
    return out


position_matrix = segment_signal_matrix(analysis_main, "position", top_segment_features)
home_away_matrix = segment_signal_matrix(
    analysis_main.assign(home_away=np.where(analysis_main["is_home"], "home", "away")),
    "home_away",
    top_segment_features,
)
checkpoint_matrix = segment_signal_matrix(analysis_main, "checkpoint", top_segment_features)
checkpoint_order = ["H1_15", "H1_30", "H1_45", "H2_15", "H2_30", "H2_45", "ET1_15"]
checkpoint_matrix = checkpoint_matrix.reindex([value for value in checkpoint_order if value in checkpoint_matrix.index])

position_matrix.to_csv(ARTIFACT_DIR / "signal_by_position.csv")
home_away_matrix.to_csv(ARTIFACT_DIR / "signal_by_home_away.csv")
checkpoint_matrix.to_csv(ARTIFACT_DIR / "signal_by_checkpoint.csv")

fig, axes = plt.subplots(3, 1, figsize=(14, 14))
plot_heatmap(axes[0], position_matrix, "Signal Stability By Position")
plot_heatmap(axes[1], home_away_matrix, "Signal Stability By Home/Away")
plot_heatmap(axes[2], checkpoint_matrix, "Signal Stability By Checkpoint")
plt.tight_layout()
plt.show()

display(Markdown("## Segment Signal Tables"))
display(Markdown("### By Position"))
display(position_matrix)
display(Markdown("### By Home/Away"))
display(home_away_matrix)
display(Markdown("### By Checkpoint"))
display(checkpoint_matrix)


## Interpretation Notes

- the shortlist is heuristic, not a final modeling decision
- high signal plus high redundancy suggests an ablation choice rather than keeping every correlated feature
- segment instability matters: a feature with strong aggregate signal but sign reversals across position or checkpoint should be treated carefully in the baseline model
- downstream modeling should still respect the leakage audit, especially the restriction on future-looking variables outside the audited `last15_*` and `cumul_*` families
